In [47]:
from dotenv import load_dotenv
import os
from supabase import create_client, Client
from pathlib import Path
import pandas as pd
from rich.progress import track, Progress
import time
from rich.console import Console
from tqdm.notebook import tqdm
import football_analytics.data.preprocessing as preprocessing
from importlib import reload
reload(preprocessing)

<module 'football_analytics.data.preprocessing' from 'C:\\Users\\david\\Documents\\Git\\football-analytics\\football_analytics\\data\\preprocessing.py'>

## Loading Supabase Info

In [3]:
# Load variables from .env into the environment
load_dotenv()

# Read variables
supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_KEY")

In [4]:
# Initialize client
supabase: Client = create_client(supabase_url, supabase_key)

In [5]:
table_name = "shots"
response = supabase.table(table_name).select("*").limit(1).execute()
print(response)

data=[{'created_at': '2025-12-02T22:20:02.042258+00:00', 'statsbomb_event_id': '4c39782f-270e-406b-a624-50c2a2e54c20', 'x1': 114.3, 'y1': 42.5, 'distance_to_goal': 6.224146527838177, 'angle_to_goal_deg': 63.495291906996705, 'keeper_distance': 5.059644256269405, 'min_defender_distance': 0.860232526704265, 'avg_defender_distance': 9.24012727409794, 'num_def_in_shot_triangle': 0, 'num_teammates_in_box': 2, 'shot_to_min_def_ratio': 7.152279806651443, 'shot_type': 'Open Play', 'body_part': 'Head', 'outcome': 'Goal', 'full_json': '{"id":"4c39782f-270e-406b-a624-50c2a2e54c20","index":1132,"period":1,"timestamp":1765153585840,"minute":26,"second":25,"type":{"id":16,"name":"Shot"},"possession":54,"possession_team":{"id":169,"name":"Bayern Munich"},"play_pattern":{"id":3,"name":"From Free Kick"},"team":{"id":169,"name":"Bayern Munich"},"duration":0.581935,"tactics":null,"related_events":["37a5aba9-b7c4-4f04-8762-a07ff5c60985","decce0ff-12b4-41d0-81ef-38c6e95cfa40"],"player":{"id":5244,"name":"Me

## Process Statsbomb Data

#### Events

In [50]:
folder_path = Path("../../data/statsbomb/open-data-master/data/events")  # or absolute path: Path("/full/path/to/data")
json_files = list(folder_path.glob("*.json"))
shots_to_upsert = []
BATCHSIZE_SHOT_UPSERT = 50
MATCH_LIMIT = 1e6

# Loop over all json files
for match_index, json_file in tqdm(enumerate(json_files), desc="Processing...", total=len(json_files)):

    match_id = json_file.stem

    df = pd.read_json(json_file)
    df_shots = df[~df['shot'].isna()].copy().reset_index(drop=True)

    # Loop over shots
    for shot_index, row in df_shots.iterrows():
        event = row
        shot_data = preprocessing.extract_shot_features(event, match_id)
        shots_to_upsert.append(shot_data)

    if len(shots_to_upsert) >= BATCHSIZE_SHOT_UPSERT:
        response = supabase.table(table_name).upsert(shots_to_upsert, on_conflict="statsbomb_event_id").execute()
        shots_to_upsert = []

    if match_index > MATCH_LIMIT:
        break


if shots_to_upsert:
    response = supabase.table(table_name).upsert(shots_to_upsert, on_conflict="statsbomb_event_id").execute()
    shots_to_upsert = []


Processing...:   0%|          | 0/3464 [00:00<?, ?it/s]

#### Competion

#### Matches

In [ ]:
supabase.table(table_name).upsert(cars_to_insert, ignore_duplicates=True).execute()